In [5]:
import cv2
import numpy as np

In [8]:
def nothing(x):
    pass

def check_if_works(var, n=0):
    if not n:
        print(var)
        n += 1


def main():
    num = 0
    video_path = 'data/test_match.mp4'
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print('Помилка: не вдалося відкрити відеофайл')
        return

    # Створення вікна
    cv2.namedWindow('Trackbars', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Trackbars', 400, 300)

    # Ініціалізація повзунків: (Назва, Вікно, Стартове значення, Максимум, Функція)
    cv2.createTrackbar('H_MIN', 'Trackbars', 0, 179, nothing)
    cv2.createTrackbar('S_MIN', 'Trackbars', 0, 255, nothing)
    cv2.createTrackbar('V_MIN', 'Trackbars', 0, 255, nothing)
    cv2.createTrackbar('H_MAX', 'Trackbars', 179, 179, nothing)
    cv2.createTrackbar('S_MAX', 'Trackbars', 255, 255, nothing)
    cv2.createTrackbar('V_MAX', 'Trackbars', 255, 255, nothing)

    paused = False

    while True:
        if not paused:
            ret, frame = cap.read()
            # check_if_works(ret, num)

            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue

            # Зменшуємо кадр, щоб влазив на екран
            frame = cv2.resize(frame, (1024, 576))
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Зчитуємо поточні значення повзунків
        h_min = cv2.getTrackbarPos('H_MIN', 'Trackbars')
        s_min = cv2.getTrackbarPos('S_MIN', 'Trackbars')
        v_min = cv2.getTrackbarPos('V_MIN', 'Trackbars')
        h_max = cv2.getTrackbarPos('H_MAX', 'Trackbars')
        s_max = cv2.getTrackbarPos('S_MAX', 'Trackbars')
        v_max = cv2.getTrackbarPos('V_MAX', 'Trackbars')

        # Формуємо масиви для маски
        # lower_yellow = np.array([14, 150, 92])
        # upper_yellow = np.array([21, 255, 242])
        # lower_yellow = np.array([h_min, s_min, v_min])
        # upper_yellow = np.array([h_max, s_max, v_max])

        lower_yellow = np.array([14, 150, 92])
        upper_yellow = np.array([21, 255, 242])
        mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)

        lower_blue = np.array([115, 60, 20])
        upper_blue = np.array([150, 220, 60])
        mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

        # mask_combined = cv2.bitwise_or(mask_yellow, mask_blue)

        mask_combined = mask_yellow

        # Очищення (Морфологія)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask_clean = cv2.erode(mask_combined, kernel, iterations=1)
        mask_clean = cv2.dilate(mask_clean, kernel, iterations=1)

        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = []

        for c in contours:
            hull = cv2.convexHull(c)
            area = cv2.contourArea(hull) # Тепер рахуємо площу ОБОЛОНКИ, а не рваної панелі

            if 50 < area < 2000:
                x_b, y_b, w_b, h_b = cv2.boundingRect(hull)
                aspect_ratio = float(w_b) / h_b

                if 0.3 < aspect_ratio < 3.0:
                    valid_contours.append(hull)

        ball_center = None
        if len(valid_contours) > 0:
            # Беремо найбільшу оболонку
            best_c = max(valid_contours, key=cv2.contourArea)

            ((x, y), radius) = cv2.minEnclosingCircle(best_c)
            M = cv2.moments(best_c)

            if M["m00"] > 0:
                ball_center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))

                cv2.circle(frame, (int(x), int(y)), int(radius), (0, 255, 255), 2)
                cv2.circle(frame, ball_center, 4, (0, 0, 255), -1)

        cv2.imshow("Cleaned Mask", mask_clean)
        cv2.imshow("Tracking", frame)

        # Накладаємо маску на оригінальний кадр для наочності
        # result = cv2.bitwise_and(frame, frame, mask=mask)

        # Вивід вікон
        cv2.imshow('Mask', mask_combined)
        # cv2.imshow('Result', result)

        key = cv2.waitKey(30) & 0xFF

        if key == ord('q'):
            print(f'lower_color = np.array([{h_min}, {s_min}, {v_min}])')
            print(f'upper_color = np.array([{h_max}, {s_max}, {v_max}])')
            break
        elif key == ord(' '):
            paused = not paused

    cap.release()
    cv2.destroyAllWindows()

main()


lower_color = np.array([0, 0, 0])
upper_color = np.array([179, 255, 255])
